基于pi电子密度的aim分析

In [1]:
from pywfn.base import Mole
from pywfn.orbtprop import obtmat
from pywfn.gridprop import CubeGrid, density
from pywfn.atomprop import charge
import numpy as np

In [2]:
mole=Mole.from_file(rf"c:\Users\11032\Desktop\gfile\pywfn\C6H6.out")
cmat_caler=obtmat.Calculator(mole)
dirs,cmat_pocv=cmat_caler.pi_pocv()
mole.set_cmat("sph",cmat_pocv)
dens_caler=density.Calculator(mole)

计算分子格点的电子密度，以电子密度>0.03为起始点筛选初猜点

In [3]:
p0,p1=mole.border()
print(p0,p1)
grider=CubeGrid()
grider.set_v1(p0,p1,0.2,4.0)
shape,grids=grider.get()
print(shape)
rhos=dens_caler.mol_rho_cm(grids,0)[0]

[-4.044571377686762, -4.6702691464564765, 0.0] [4.044571377686762, 4.6702691464564765, 0.0]
[81, 87, 40]


In [4]:
def find(r0,k=1e-8):
    for i in range(100):
        val, grad, hess = dens_caler.mol_rho_cm(r0.reshape(1,3), 2)
        grad = grad[0]
        hess = hess[0]

        norm = np.linalg.norm(grad)
        if norm < 1e-10:
            break

        eigvals = np.linalg.eigvals(hess)
        eigmin = eigvals.min()
        if np.abs(eigmin) < k:
            lambda_reg = -eigmin + k
            hess += lambda_reg * np.eye(3)

        try:
            hess_inv=np.linalg.inv(hess)
        except np.linalg.LinAlgError:
            k*=10
            continue
        dr=hess_inv@grad.reshape(3,1)
        dl=np.linalg.norm(dr)
        if dl>0.1: # 限制步长
            dr=dr/dl*0.1

        r0 -= dr.flatten()
    pnum=0
    nnum=0
    for eigv in eigvals:
        if eigv>1e-6:
            pnum+=1
        if eigv<-1e-6:
            nnum+=1
    return r0,val,pnum-nnum

In [6]:
r0s=np.array(grids)
r0s.flags.writeable=False
rfs=[]
cpts=[]
vals=[]
idxs=np.where(rhos>0.03)
for row in idxs[0]:
    r0=r0s[row]
    rf,val,cpt=find(r0.copy())
    if np.linalg.norm(rf)>10:
        continue
    rfs.append(rf)
    vals.append(val)
    cpts.append(cpt)
    # print(r0s[row],r0)
    print(f"\r{row}/{len(r0s)}:{r0}->{rf}",end="")
    # break
rfs=np.array(rfs)
cpts=np.array(cpts)
vals=np.array(vals)
np.save("./r0s.npy",r0s)
np.save("./rfs.npy",rfs)
np.save("./vals.npy",vals)
np.save("./cpts.npy",cpts)

196903/281880:[3.15542862 1.32973085 0.6       ]->[9.3000107  0.90328883 0.11773082]73082]7117e+00]01]

In [8]:
list(cpts)

[np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(-1),
 np.int64(0),
 np.int64(-1),
 np.int64(-1),
 np.int64(0),
 np.int64(-1),
 np.int64(-1),
 np.int64(0),
 np.int64(-1),
 np.int64(-1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 n